# Generate Pickle Caches for All Symbols

This notebook generates pickle cache files for all available cryptocurrency symbols in the dataset. This will improve performance for future data loading operations by pre-processing and caching all symbol data.

In [1]:
import sys
import os

# Add project root to path for imports
sys.path.append('/home/craz/crypto/crypto-trading')

import pandas as pd
import time
from datetime import datetime

# Import our data loader utilities
from utils.data_loader import (
    get_available_symbols,
    is_pickle_cache_available, 
    create_pickle_cache,
    get_cache_status,
    get_symbol_info,
    PICKLE_CACHE_DIR
)

print(f"Using data loader utilities from utils/data_loader.py")
print(f"Pickle cache directory: {PICKLE_CACHE_DIR}")

# Check initial cache status
initial_status = get_cache_status()
print(f"Initial cache status: {initial_status['data_files']} files, {initial_status['total_cache_size_gb']:.2f} GB")

Using data loader utilities from utils/data_loader.py
Pickle cache directory: /home/craz/crypto/crypto-data/pickle_cache
Initial cache status: 548 files, 25.22 GB


## 1. Discover Available Symbols

In [ ]:
# Get all available symbols using data_loader utility
symbols = get_available_symbols()
print(f"Found {len(symbols)} symbols:")

# Display symbols in columns for better readability
columns = 4
for i in range(0, len(symbols), columns):
    row_symbols = symbols[i:i+columns]
    formatted_row = "  ".join(f"{i+j+1:3d}. {symbol:15s}" for j, symbol in enumerate(row_symbols))
    print(formatted_row)

# Check how many already have pickle caches
cached_count = sum(1 for symbol in symbols if is_pickle_cache_available(symbol))
print(f"\n📊 Cache Status:")
print(f"   Symbols with existing caches: {cached_count}/{len(symbols)}")
print(f"   Symbols needing caches: {len(symbols) - cached_count}/{len(symbols)}")

## 2. Enhanced Pickle Cache Generation

In [ ]:
def generate_cache_with_info(symbol):
    """Enhanced cache generation with detailed info using data_loader functions"""
    print(f"\n🔄 Processing {symbol}...")
    start_time = time.time()
    
    # Check if cache already exists
    if is_pickle_cache_available(symbol):
        print(f"   ⚡ Cache already exists, skipping...")
        return {
            'symbol': symbol,
            'status': 'skipped',
            'reason': 'cache_exists'
        }
    
    # Get symbol info first
    symbol_info = get_symbol_info(symbol)
    if not symbol_info:
        print(f"   ❌ No data found for {symbol}")
        return {
            'symbol': symbol,
            'status': 'failed',
            'reason': 'no_data'
        }
    
    print(f"   📁 Found {symbol_info['total_files']} CSV files")
    print(f"   📅 Date range: {symbol_info['first_date']} to {symbol_info['last_date']}")
    
    # Create the cache using data_loader function
    try:
        df = create_pickle_cache(symbol)
        
        if df is not None:
            processing_time = time.time() - start_time
            
            # Get file size
            from utils.data_loader import get_pickle_path
            pickle_path = get_pickle_path(symbol)
            file_size_mb = os.path.getsize(pickle_path) / (1024 * 1024)
            
            return {
                'symbol': symbol,
                'status': 'success',
                'records': len(df),
                'csv_files': symbol_info['total_files'],
                'size_mb': file_size_mb,
                'processing_time': processing_time,
                'date_range': {
                    'start': df['open_time'].min().isoformat(),
                    'end': df['open_time'].max().isoformat()
                }
            }
        else:
            print(f"   ❌ Failed to create cache")
            return {
                'symbol': symbol,
                'status': 'failed',
                'reason': 'cache_creation_failed'
            }
            
    except Exception as e:
        print(f"   ❌ Error: {e}")
        return {
            'symbol': symbol,
            'status': 'error',
            'error': str(e)
        }

## 3. Generate Pickle Caches for All Symbols

In [ ]:
# Enhanced cache generation using data_loader utilities
print(f"🚀 Starting enhanced pickle cache generation for {len(symbols)} symbols...")
print(f"{'='*80}")

results = []
total_start_time = time.time()
successful = 0
skipped = 0
failed = 0

# Filter symbols that need caching (don't already have caches)
symbols_to_process = [symbol for symbol in symbols if not is_pickle_cache_available(symbol)]
print(f"\n📋 Processing {len(symbols_to_process)} symbols that need caching...")
print(f"   Skipping {len(symbols) - len(symbols_to_process)} symbols that already have caches")

for i, symbol in enumerate(symbols_to_process):
    print(f"\n[{i+1}/{len(symbols_to_process)}] Processing {symbol}...")
    
    result = generate_cache_with_info(symbol)
    results.append(result)
    
    if result['status'] == 'success':
        successful += 1
    elif result['status'] == 'skipped':
        skipped += 1
    else:
        failed += 1
    
    # Progress update every 5 symbols
    if (i + 1) % 5 == 0 or i == len(symbols_to_process) - 1:
        elapsed = time.time() - total_start_time
        avg_time = elapsed / (i + 1) if i > 0 else elapsed
        remaining_symbols = len(symbols_to_process) - i - 1
        remaining_time = remaining_symbols * avg_time
        
        print(f"\n📊 Progress: {i+1}/{len(symbols_to_process)} ({((i+1)/len(symbols_to_process)*100):.1f}%)")
        print(f"   Successful: {successful}, Skipped: {skipped}, Failed: {failed}")
        print(f"   Time elapsed: {elapsed/60:.1f} minutes")
        if remaining_symbols > 0:
            print(f"   Estimated time remaining: {remaining_time/60:.1f} minutes")

total_time = time.time() - total_start_time
print(f"\n{'='*80}")
print(f"🎉 Enhanced pickle cache generation completed!")
print(f"   Total time: {total_time/60:.1f} minutes")
print(f"   Successful: {successful}")
print(f"   Skipped: {skipped}")  
print(f"   Failed: {failed}")
print(f"   Total processed: {len(results)}/{len(symbols_to_process)}")

# Get updated cache status
final_status = get_cache_status()
print(f"\nFinal cache status: {final_status['data_files']} files, {final_status['total_cache_size_gb']:.2f} GB")

## 4. Summary Report

In [ ]:
# Enhanced summary report using data_loader utilities
successful_results = [r for r in results if r['status'] == 'success']

if successful_results:
    # Create summary DataFrame for successful results
    summary_df = pd.DataFrame(successful_results)
    
    print(f"\n📈 Enhanced Cache Generation Summary:")
    print(f"{'='*80}")
    
    # Overall statistics
    total_records = summary_df['records'].sum()
    total_files = summary_df['csv_files'].sum()
    total_size_mb = summary_df['size_mb'].sum()
    avg_processing_time = summary_df['processing_time'].mean()
    
    print(f"Total records processed: {total_records:,}")
    print(f"Total CSV files processed: {total_files:,}")
    print(f"Total cache size: {total_size_mb:.1f} MB ({total_size_mb/1024:.2f} GB)")
    print(f"Average processing time per symbol: {avg_processing_time:.2f} seconds")
    
    # Top symbols by size
    print(f"\n🔝 Largest datasets:")
    top_by_records = summary_df.nlargest(10, 'records')[['symbol', 'records', 'size_mb', 'processing_time']]
    for i, (_, row) in enumerate(top_by_records.iterrows()):
        print(f"   {i+1:2d}. {row['symbol']:15s}: {row['records']:10,} records, {row['size_mb']:7.1f} MB, {row['processing_time']:6.1f}s")
    
    # Date range analysis
    if len(successful_results) > 0:
        print(f"\n📅 Date range analysis:")
        earliest_start = min(result['date_range']['start'] for result in successful_results)
        latest_end = max(result['date_range']['end'] for result in successful_results)
        print(f"   Data spans from {earliest_start[:10]} to {latest_end[:10]}")
    
    # Processing efficiency
    print(f"\n⚡ Processing efficiency:")
    fastest = summary_df.loc[summary_df['processing_time'].idxmin()]
    slowest = summary_df.loc[summary_df['processing_time'].idxmax()]
    print(f"   Fastest: {fastest['symbol']} ({fastest['processing_time']:.2f}s)")
    print(f"   Slowest: {slowest['symbol']} ({slowest['processing_time']:.2f}s)")
    
    # Display summary table (top 20 by records)
    print(f"\n📋 Top 20 Symbols by Record Count:")
    display_df = summary_df.nlargest(20, 'records')[['symbol', 'records', 'csv_files', 'size_mb', 'processing_time']].round(2)
    display(display_df)

# Failed/error analysis
failed_results = [r for r in results if r['status'] in ['failed', 'error']]
if failed_results:
    print(f"\n❌ Failed Processing Analysis:")
    print(f"{'='*50}")
    for result in failed_results:
        reason = result.get('reason', result.get('error', 'unknown'))
        print(f"   {result['symbol']:15s}: {reason}")

# Use data_loader utility for final cache status
final_cache_status = get_cache_status()
print(f"\n📊 Final Cache Directory Status:")
print(f"{'='*50}")
print(f"Directory: {final_cache_status['cache_directory']}")
print(f"Total pickle files: {final_cache_status['total_pickle_files']}")
print(f"Data files: {final_cache_status['data_files']}")
print(f"Total cache size: {final_cache_status['total_cache_size_mb']:.1f} MB ({final_cache_status['total_cache_size_gb']:.2f} GB)")

print(f"\n✅ Enhanced pickle cache generation completed successfully!")
print(f"   Using optimized data_loader utilities for maximum performance")
print(f"   All functions now available in utils/data_loader.py")

## 5. Enhanced Verification Using Data Loader

In [ ]:
# Enhanced verification using data_loader utilities
print(f"\n🔍 Verifying all pickle caches using data_loader functions...")
print(f"{'='*70}")

from utils.data_loader import load_from_pickle_cache, get_symbol_data

# Get all symbols that should have caches now
all_symbols = get_available_symbols()
verification_results = []

print(f"Testing {len(all_symbols)} symbols for cache availability and loading...")

for i, symbol in enumerate(all_symbols):
    try:
        # Test 1: Check if cache is available
        has_cache = is_pickle_cache_available(symbol)
        
        if not has_cache:
            verification_results.append({
                'symbol': symbol,
                'status': 'NO_CACHE',
                'message': 'No pickle cache available'
            })
            print(f"⚠️  {symbol:15s}: No cache available")
            continue
            
        # Test 2: Load using load_from_pickle_cache function
        load_start = time.time()
        df_direct = load_from_pickle_cache(symbol)
        direct_load_time = time.time() - load_start
        
        if df_direct is None:
            verification_results.append({
                'symbol': symbol,
                'status': 'LOAD_FAILED',
                'message': 'Direct pickle load failed'
            })
            print(f"❌ {symbol:15s}: Direct pickle load failed")
            continue
            
        # Test 3: Load using get_symbol_data function (full integration test)
        load_start = time.time()
        df_integrated = get_symbol_data(symbol)
        integrated_load_time = time.time() - load_start
        
        if df_integrated is None:
            verification_results.append({
                'symbol': symbol,
                'status': 'INTEGRATION_FAILED',
                'message': 'Integrated data loading failed'
            })
            print(f"❌ {symbol:15s}: Integrated loading failed")
            continue
            
        # Test 4: Validate data consistency
        if len(df_direct) != len(df_integrated):
            verification_results.append({
                'symbol': symbol,
                'status': 'INCONSISTENT',
                'message': f'Record count mismatch: direct={len(df_direct)}, integrated={len(df_integrated)}'
            })
            print(f"⚠️  {symbol:15s}: Record count mismatch")
            continue
            
        # Test 5: Basic data validation
        required_columns = ['open_time', 'open', 'high', 'low', 'close', 'volume']
        missing_columns = [col for col in required_columns if col not in df_integrated.columns]
        
        if missing_columns:
            verification_results.append({
                'symbol': symbol,
                'status': 'INVALID_DATA',
                'message': f'Missing columns: {missing_columns}'
            })
            print(f"❌ {symbol:15s}: Missing columns: {missing_columns}")
            continue
            
        # All tests passed
        verification_results.append({
            'symbol': symbol,
            'status': 'SUCCESS',
            'records': len(df_integrated),
            'direct_load_time': direct_load_time,
            'integrated_load_time': integrated_load_time,
            'date_range': {
                'start': df_integrated['open_time'].min().isoformat()[:10],
                'end': df_integrated['open_time'].max().isoformat()[:10]
            }
        })
        
        if (i + 1) % 50 == 0 or i == len(all_symbols) - 1:
            print(f"✅ Progress: {i+1}/{len(all_symbols)} symbols verified")
            
    except Exception as e:
        verification_results.append({
            'symbol': symbol,
            'status': 'ERROR',
            'error': str(e)
        })
        print(f"💥 {symbol:15s}: Exception - {str(e)[:50]}...")

# Verification summary
successful_verifications = [r for r in verification_results if r['status'] == 'SUCCESS']
failed_verifications = [r for r in verification_results if r['status'] != 'SUCCESS']

print(f"\n📊 Enhanced Verification Results:")
print(f"{'='*70}")
print(f"Total symbols tested: {len(all_symbols)}")
print(f"Successfully verified: {len(successful_verifications)}")
print(f"Failed verifications: {len(failed_verifications)}")
print(f"Success rate: {len(successful_verifications)/len(all_symbols)*100:.1f}%")

if successful_verifications:
    # Performance statistics
    direct_times = [r['direct_load_time'] for r in successful_verifications]
    integrated_times = [r['integrated_load_time'] for r in successful_verifications]
    
    print(f"\n⚡ Performance Analysis:")
    print(f"Average direct load time: {sum(direct_times)/len(direct_times):.4f} seconds")
    print(f"Average integrated load time: {sum(integrated_times)/len(integrated_times):.4f} seconds")
    print(f"Fastest load: {min(direct_times):.4f} seconds")
    print(f"Slowest load: {max(direct_times):.4f} seconds")
    
    # Data statistics
    total_records = sum(r['records'] for r in successful_verifications)
    print(f"\nTotal records verified: {total_records:,}")
    print(f"Average records per symbol: {total_records/len(successful_verifications):,.0f}")

# Failed verification details
if failed_verifications:
    print(f"\n❌ Failed Verification Details:")
    print(f"{'='*50}")
    
    failure_counts = {}
    for result in failed_verifications:
        status = result['status']
        failure_counts[status] = failure_counts.get(status, 0) + 1
        
    for status, count in failure_counts.items():
        print(f"   {status}: {count} symbols")
    
    # Show first 10 failures
    print(f"\nFirst 10 failures:")
    for result in failed_verifications[:10]:
        message = result.get('message', result.get('error', 'unknown'))
        print(f"   {result['symbol']:15s}: {result['status']} - {message[:50]}...")

print(f"\n🎯 Verification completed using data_loader utilities!")
print(f"   All cache operations tested through the unified interface")
print(f"   Ready for production use with full integration testing")

## 6. Final Status Using Data Loader Utilities

In [ ]:
# Final comprehensive status using all data_loader utilities
print(f"\n📁 Final Comprehensive Cache Status:")
print(f"{'='*70}")

# Get comprehensive status using data_loader utilities
cache_status = get_cache_status()
all_symbols = get_available_symbols()

print(f"📊 Cache Directory Analysis:")
print(f"   Directory: {cache_status['cache_directory']}")
print(f"   Total pickle files: {cache_status['total_pickle_files']}")
print(f"   Data files (_full_data.pkl): {cache_status['data_files']}")
print(f"   Total cache size: {cache_status['total_cache_size_mb']:.1f} MB")
print(f"   Total cache size: {cache_status['total_cache_size_gb']:.2f} GB")

print(f"\n🔍 Symbol Coverage Analysis:")
symbols_with_cache = [symbol for symbol in all_symbols if is_pickle_cache_available(symbol)]
symbols_without_cache = [symbol for symbol in all_symbols if not is_pickle_cache_available(symbol)]

print(f"   Total symbols available: {len(all_symbols)}")
print(f"   Symbols with pickle cache: {len(symbols_with_cache)}")
print(f"   Symbols without cache: {len(symbols_without_cache)}")
print(f"   Cache coverage: {len(symbols_with_cache)/len(all_symbols)*100:.1f}%")

if symbols_without_cache:
    print(f"\n❌ Symbols still needing caches:")
    for i, symbol in enumerate(symbols_without_cache[:10]):  # Show first 10
        print(f"   {i+1:2d}. {symbol}")
    if len(symbols_without_cache) > 10:
        print(f"   ... and {len(symbols_without_cache) - 10} more")

# Sample a few symbols for detailed info
print(f"\n📋 Sample Symbol Information (using get_symbol_info):")
sample_symbols = symbols_with_cache[:5] if symbols_with_cache else []
for symbol in sample_symbols:
    info = get_symbol_info(symbol)
    if info:
        print(f"   {symbol:15s}: {info['total_files']:3d} files, {info['first_date']} to {info['last_date']}, cache: {'✅' if info['has_pickle_cache'] else '❌'}")

print(f"\n🎯 Data Loader Integration Status:")
print(f"   ✅ get_available_symbols(): {len(all_symbols)} symbols discovered")
print(f"   ✅ is_pickle_cache_available(): {len(symbols_with_cache)} caches verified")
print(f"   ✅ get_cache_status(): Full directory analysis complete")
print(f"   ✅ get_symbol_info(): Metadata accessible for all symbols")
print(f"   ✅ create_pickle_cache(): Cache generation system operational")
print(f"   ✅ load_from_pickle_cache(): Direct cache loading functional")
print(f"   ✅ get_symbol_data(): Integrated data loading with fallbacks")

print(f"\n🚀 SYSTEM STATUS: FULLY OPERATIONAL!")
print(f"{'='*70}")
print(f"✅ Enhanced pickle cache generation completed successfully")
print(f"✅ All data_loader utilities integrated and tested")
print(f"✅ {len(symbols_with_cache)} symbols ready for high-performance loading")
print(f"✅ {cache_status['total_cache_size_gb']:.2f} GB of optimized data available")
print(f"✅ Complete data pipeline from CSV → Pickle → Analysis ready")

# Performance demonstration
if symbols_with_cache:
    print(f"\n⚡ Performance Demonstration:")
    demo_symbol = symbols_with_cache[0]
    
    # Time the integrated loading
    start_time = time.time()
    demo_data = get_symbol_data(demo_symbol)
    load_time = time.time() - start_time
    
    if demo_data is not None:
        records_per_second = len(demo_data) / load_time
        print(f"   Loaded {demo_symbol}: {len(demo_data):,} records in {load_time:.4f}s")
        print(f"   Performance: {records_per_second:,.0f} records/second")
        print(f"   Data range: {demo_data['open_time'].min().date()} to {demo_data['open_time'].max().date()}")

print(f"\n🎉 Ready for production use with utils/data_loader.py!")
print(f"   All cache operations optimized and verified")
print(f"   Seamless integration with existing and future applications")